# 04 — Model Training

This notebook trains all four classification models.

**Objectives:**
- Load and preprocess the dataset
- Split into train/test (80/20, stratified)
- Train DummyClassifier, MultinomialNB, LogisticRegression, LinearSVC
- Save trained pipelines

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import time

from src.data_loader import load_and_merge_data, remove_duplicates
from src.preprocessing import clean_text, combine_title_text
from src.train import (
    build_pipelines, prepare_train_test_split,
    train_all_models, save_pipeline, save_training_results, TFIDF_PARAMS
)
print("Libraries loaded.")
print(f"TF-IDF parameters: {TFIDF_PARAMS}")

## 1. Load and Prepare Data

In [ ]:
df = load_and_merge_data()
print(f"Loaded: {len(df)} articles")

df, n_removed = remove_duplicates(df, subset=["title", "text"])
print(f"Removed {n_removed} duplicates. Remaining: {len(df)}")

df["combined_text"] = df.apply(
    lambda row: combine_title_text(row.get("title", ""), row.get("text", "")), axis=1
)
df["clean_text"] = df["combined_text"].apply(
    lambda x: clean_text(x, remove_source_markers=True)
)

empty_mask = df["clean_text"].str.strip() == ""
n_empty = empty_mask.sum()
if n_empty > 0:
    df = df[~empty_mask].reset_index(drop=True)
    print(f"Removed {n_empty} empty articles.")
print(f"Dataset ready: {len(df)} articles")

## 2. Train/Test Split

80/20 stratified, random_state=42

In [ ]:
X_train, X_test, y_train, y_test = prepare_train_test_split(
    df, text_column="clean_text", label_column="label",
    test_size=0.2, random_state=42
)

## 3. Build and Train All Models

In [ ]:
pipelines = build_pipelines()
print("Models to train:")
for name in pipelines:
    print(f"  - {name}")

model_results = train_all_models(X_train, y_train, pipelines)

## 4. Training Summary

In [ ]:
print("\nTraining Times:")
print(f"{'Model':<25} {'Time (s)':<15}")
print("-" * 40)
for name, info in model_results.items():
    print(f"{name:<25} {info['train_time']:>10.2f}s")

## 5. Save Models

In [ ]:
save_training_results(model_results)
for name, info in model_results.items():
    safe = name.replace(" ", "_").lower()
    save_pipeline(info["pipeline"], f"{safe}.joblib")
print("\nAll models trained and saved.")

## 6. Quick Verification

In [ ]:
test_text = "The president announced a new trade agreement with European leaders today."
for name, info in model_results.items():
    pred = info["pipeline"].predict([test_text])[0]
    label = "Real" if pred == 1 else "Fake"
    print(f"{name}: predicted {label} (label={pred})")